# 01 · Dataset Overview — Priority Class Mapping

**Goal:** inspect `jason23322/high-accuracy-email-classifier` and decide how its categories map to PriorMail's 4 urgency classes (`urgent | high | normal | low`).

This is the **Step 1–5** workflow from the mapping guide. The mapping itself is owned by **Insan + Faiz** (`docs/ML_PIPELINE.md` §2) — this notebook only *informs* it.

> ⚠️ Exploration only (CLAUDE.md §6). Do **not** import anything from here into `src/`. Reading `src` constants (one-way) is fine and intended.
>
> 🧹 **Strip outputs before committing** (`nbstripout`, installed via `[dev]`).

**Run order:**
1. Setup & imports
2. Load the dataset, see splits + columns
3. The exact category strings + per-class counts *(copy these into `loaders.py`)*
4. Read sample emails per category — map on **urgency**, not the category name
5. Draft the mapping (scratch — final version goes in `src/data/loaders.py`)
6. Check the resulting 4-class distribution

## 1 · Setup

Run this on a box with network (Colab/Kaggle or local with deps). Adds the repo root to `sys.path` so we can read the **target labels** straight from the contract (`src/utils/constants.py`) — keeps this notebook in sync if the enum ever changes.

In [7]:
import sys
from pathlib import Path

# Make the repo root importable so we can read target labels from the contract.
# (Reading src is one-way and fine; never import notebook code back into src — CLAUDE.md §6.)
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from datasets import load_dataset

from src.utils.constants import PRIORITY_LABELS

DATASET_ID = "jason23322/high-accuracy-email-classifier"
print("Target priority classes (the contract):", PRIORITY_LABELS)

Target priority classes (the contract): ('urgent', 'high', 'normal', 'low')


## 2 · Load the dataset

Inspect splits, columns, and one example. **Check the printed column names** — the rest of the notebook assumes a label column and a text/subject column; set `LABEL_COL` / `TEXT_COL` below to match what you actually see.

In [8]:
ds = load_dataset(DATASET_ID)
print(ds)  # splits, num_rows, column names

split = "train" if "train" in ds else list(ds.keys())[0]
print(f"\nUsing split: {split!r}")
print("Columns:", ds[split].column_names)
print("\nFirst example:")
ds[split][0]

train.json:   0%|          | 0.00/4.15M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'subject', 'body', 'text', 'category', 'category_id'],
        num_rows: 10780
    })
    test: Dataset({
        features: ['id', 'subject', 'body', 'text', 'category', 'category_id'],
        num_rows: 2697
    })
})

Using split: 'train'
Columns: ['id', 'subject', 'body', 'text', 'category', 'category_id']

First example:


{'id': 'promotions_582',
 'subject': 'Anniversary Special: Buy one get one free',
 'body': 'As our loyal customer, get exclusive $60 off $75+: example.com/6058 Offer code: WELCOME20.',
 'text': 'Anniversary Special: Buy one get one free As our loyal customer, get exclusive $60 off $75+: example.com/6058 Offer code: WELCOME20.',
 'category': 'promotions',
 'category_id': 1}

In [12]:
# --- ADJUST to match the column names printed above ---
LABEL_COL = "category"   # column holding the source category
TEXT_COL = "body"     # column holding the email text (or "subject"/"body")

assert LABEL_COL in ds[split].column_names, f"{LABEL_COL!r} not in {ds[split].column_names}"
assert TEXT_COL in ds[split].column_names, f"{TEXT_COL!r} not in {ds[split].column_names}"

# If the label column is a typed ClassLabel, this reveals the human-readable names.
feat = ds[split].features[LABEL_COL]
print("Label feature:", feat)
LABEL_NAMES = getattr(feat, "names", None)
print("ClassLabel names:", LABEL_NAMES)  # None if labels are plain strings

Label feature: Value('string')
ClassLabel names: None


## 3 · Category strings + counts

The output here is what you **copy verbatim** into the `PRIORITY_CLASS_MAPPING` keys in `src/data/loaders.py`. Casing matters — they're dict keys. The counts also flag imbalance (feeds the `class_weights: balanced` setting in the config).

In [13]:
# Normalize labels to human-readable strings whether they're ClassLabel ints or plain strings.
raw_labels = ds[split][LABEL_COL]
if LABEL_NAMES is not None:
    str_labels = [LABEL_NAMES[i] for i in raw_labels]
else:
    str_labels = [str(x) for x in raw_labels]

counts = pd.Series(str_labels).value_counts()
print("Source categories and counts:\n")
print(counts)
print(f"\n{len(counts)} distinct categories.")

# Copy-paste-ready stub for loaders.py — fill the right-hand side after Step 4.
print("\n--- paste into src/data/loaders.py, then assign each to urgent/high/normal/low ---")
for cat in counts.index:
    print(f'    {cat!r}: "",')

Source categories and counts:

forum           1800
verify_code     1800
promotions      1796
social_media    1796
spam            1794
updates         1794
Name: count, dtype: int64

6 distinct categories.

--- paste into src/data/loaders.py, then assign each to urgent/high/normal/low ---
    'forum': "",
    'verify_code': "",
    'promotions': "",
    'social_media': "",
    'spam': "",
    'updates': "",


## 4 · Read samples per category

For each source category, read ~15 emails and ask: **if this hit a user's inbox, what urgency would a person assign?** Map on that answer, not the category name. Watch for categories that genuinely split across two urgencies (e.g. "updates" = a receipt *or* a security alert) — pick the majority and note the compromise.

In [14]:
SAMPLES_PER_CATEGORY = 15
TEXT_PREVIEW_CHARS = 200

# Pair each row's text with its string label, then group.
paired = list(zip(str_labels, ds[split][TEXT_COL]))

for cat in counts.index:
    print(f"\n{'=' * 70}\n  {cat}  (n={counts[cat]})\n{'=' * 70}")
    shown = 0
    for lbl, text in paired:
        if lbl != cat:
            continue
        preview = " ".join(str(text).split())[:TEXT_PREVIEW_CHARS]
        print(f"  • {preview}")
        shown += 1
        if shown >= SAMPLES_PER_CATEGORY:
            break


  forum  (n=1800)
  • Trending: "cooking" (258 comments). View: support.site/ticket/456.
  • Thread "Discussion 719" moved from General to Support. Continue: support.site/ticket/456.
  • Thread: "Discussion 593" started by PixelArtist. Settings: support.site/ticket/456.
  • {event} starts {time}. Prepare: support.site/ticket/456.
  • User: TechGuru99 post. support.site/ticket/456.
  • Thread "Discussion 551" moved from General to Support. Continue: support.site/ticket/456.
  • You earned "Achievement 3" badge. View profile: wiki.site/page/789.
  • Thread "Discussion 232" moved from General to Support. Continue: forum.com/thread/123.
  • Thread "Discussion 958" moved from General to Support. Continue: forum.com/thread/123.
  • {event} starts {time}. Prepare: forum.com/thread/123.
  • report per report #1093. Details: wiki.site/page/789.
  • Thread: "Discussion 574" started by PixelArtist. Settings: forum.com/thread/123.
  • User: PixelArtist posted helpful solution. View: wiki.site/pag

## 5 · Draft the mapping (scratch)

Fill in `DRAFT_MAPPING` below using what you saw in Step 4. This is a **scratch pad** — the authoritative copy lives in `src/data/loaders.py`, reviewed by Faiz before it's final (`docs/ML_PIPELINE.md` §2). The asserts here catch the two common mistakes: a typo'd target label, and a category left unmapped.

In [17]:
# Map each SOURCE category (left) to one PriorMail class (right): urgent | high | normal | low
# Example values are placeholders — replace with your Step 4 judgments.
DRAFT_MAPPING = {
    # cat: "urgent" | "high" | "normal" | "low"
    "forum": "normal",
    "verify_code": "urgent",
    "promotions": "low",
    "social_media": "normal",
    "spam": "low",
    "updates": "high",
}

source_cats = set(counts.index)
mapped_cats = set(DRAFT_MAPPING)

bad_targets = {v for v in DRAFT_MAPPING.values() if v not in PRIORITY_LABELS}
assert not bad_targets, f"Not valid priority labels: {bad_targets} (allowed: {PRIORITY_LABELS})"

missing = source_cats - mapped_cats
extra = mapped_cats - source_cats
assert not missing, f"Categories with no mapping (would KeyError at load): {missing}"
assert not extra, f"Mapped keys not present in the dataset (typo?): {extra}"

print("✓ Draft mapping is complete and valid.")
DRAFT_MAPPING

✓ Draft mapping is complete and valid.


{'forum': 'normal',
 'verify_code': 'urgent',
 'promotions': 'low',
 'social_media': 'normal',
 'spam': 'low',
 'updates': 'high'}

## 6 · Resulting 4-class distribution

Apply the draft mapping and check the post-mapping balance. If one class dominates (e.g. >80% `low`) or a class is nearly empty (likely `urgent`), the §8 gate **per-class recall ≥ 0.65** will be hard — revisit Step 5, or plan to lean on the internal labeled set (§7) for the starved classes.

In [18]:
# Requires a complete DRAFT_MAPPING from Step 5.
mapped = pd.Series([DRAFT_MAPPING[c] for c in str_labels])
dist = mapped.value_counts().reindex(list(PRIORITY_LABELS), fill_value=0)
pct = (dist / dist.sum() * 100).round(1)

summary = pd.DataFrame({"count": dist, "pct": pct})
print(summary)

if (pct < 5).any():
    starved = list(pct[pct < 5].index)
    print(f"\n⚠️  Under-represented (<5%): {starved} — recall ≥ 0.65 will be hard here.")
if (pct > 80).any():
    print(f"\n⚠️  One class dominates (>80%) — consider the mapping or resampling.")

        count   pct
urgent   1800  16.7
high     1794  16.6
normal   3596  33.4
low      3590  33.3
